# 02b - HH-RLHF DPO with a stronger learning rate (documented deviation)
Guide LR 1e-6 gave a small reward margin (0.21) and almost no change in refusals, so this run uses LR 5e-6 with everything else unchanged. It then generates its own evaluation outputs with the same decoding settings as notebook 05. Attach outputs of notebooks 00, 01 and 05.

In [ ]:
# ---- Task 1: repository + environment setup (Kaggle, GPU T4 x2) ----
# Get the code: preferably from the private Kaggle dataset "advanced-genai-pa2-code"
# (attached as input); otherwise clone the private GitHub repo using a Kaggle secret
# named GITHUB_TOKEN (Add-ons -> Secrets).
REPO_URL = "https://github.com/zainabusman22/advanced-genai-pa2.git"
REPO = "/kaggle/working/advanced-genai-pa2"

import glob, os, shutil, subprocess, sys
if not os.path.exists(REPO):
    code = [os.path.dirname(f) for f in glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True)
            if "advanced-genai-pa2-code" in f]
    if code:
        shutil.copytree(code[0], REPO)
        print("code copied from", code[0])
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        authed = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
        subprocess.run(["git", "clone", "--depth", "1", authed, REPO], check=True, capture_output=True)
        subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)
        del token, authed
os.chdir(REPO)
print(open("CODE_VERSION.txt").read() if os.path.exists("CODE_VERSION.txt") else
      subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
!nvidia-smi

In [ ]:
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "Select the GPU T4 x2 accelerator"

In [ ]:
# Locate artifacts from earlier notebooks attached as input datasets.
import glob
def find_dir(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"nothing matching {pattern} under /kaggle/input - attach the previous notebook's output")
    print(pattern, "->", hits[0])
    return os.path.dirname(hits[0])

In [ ]:
SFT_BEST = find_dir("sft/best/config.json")
DATA = find_dir("data/processed/hh_train.jsonl")
GENS = find_dir("generations/hh/sft.json")  # SFT generations from notebook 05
!mkdir -p outputs/generations && cp -rn {GENS}/../* outputs/generations/ && ls -R outputs/generations

In [ ]:
!python scripts/train_dpo.py --sft_path {SFT_BEST} \
    --train_path {DATA}/hh_train.jsonl --val_path {DATA}/hh_val.jsonl \
    --output_dir outputs/dpo_hh_lr5e-6 --beta 0.1 --lr 5e-6 --grad_accum 32
assert os.path.exists("outputs/dpo_hh_lr5e-6/best/config.json"), "training failed"

In [ ]:
import subprocess
G, M = "python scripts/generate.py", "outputs/dpo_hh_lr5e-6/best"
jobs = [f"{G} --device cuda:0 --model_path {M} --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator_name qwen2.5-0.5b-dpo-hh-lr5e-6 --output_path outputs/generations/hh/dpo_hh_lr5e-6.json",
        f"{G} --device cuda:1 --model_path {M} --eval_set eval_sets/alpaca_eval_300_seed42.json --generator_name qwen2.5-0.5b-dpo-hh-lr5e-6 --dataset_name alpaca_eval --output_path outputs/generations/alpaca/dpo_hh_lr5e-6.json"]
procs = [subprocess.Popen(c, shell=True, stdout=open(f"/kaggle/working/gen{i}.log", "w"), stderr=subprocess.STDOUT) for i, c in enumerate(jobs)]
print([p.wait() for p in procs])
!tail -2 /kaggle/working/gen0.log /kaggle/working/gen1.log
!python scripts/build_pairs.py --model_path outputs/generations/hh/dpo_hh_lr5e-6.json --baseline_path outputs/generations/hh/sft.json --eval_kind safety --output_path outputs/eval/hh_dpo_lr5e-6_vs_sft_pairs.jsonl
!python scripts/build_pairs.py --model_path outputs/generations/alpaca/dpo_hh_lr5e-6.json --baseline_path outputs/generations/alpaca/sft.json --eval_kind general --output_path outputs/eval/xhh_dpo_lr5e-6_vs_sft_alpaca_pairs.jsonl

In [ ]:
# Small archive of metrics/configs/generations (model weights are already kept as notebook output;
# zipping them too would double disk use).
!cd {REPO} && zip -qr /kaggle/working/dpo_hh_lr5e-6_outputs.zip outputs/dpo_hh_lr5e-6 outputs/generations outputs/eval -x "*.safetensors" "*.bin" "*/trainer/*"
!ls -lh /kaggle/working/*.zip